# Setup and Configurations
Import libraries, configure which data sources to include, and set global pipeline flags.

| Flag | Effect |
|---|---|
| `DATA_SOURCE` | Which generators to load (`"hermes"`, `"haiku"`, `"deepseek"`) |
| `PSEUDO_LABEL` | Load pseudolabeled CSVs instead of raw generation outputs |
| `COMBINE_SAMPLES` | Concatenate 10 individual reports into composite long-document samples |
| `ADD_FEW_SHOT_EXAMPLES` | Inject in-context examples into a random 10% of training rows |

In [ ]:
import json
import random
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

from pipeline.data.preprocessing import preprocess_text
from pipeline.data.augmentator import *
from pipeline.data.augmentations import *
from combine_utils import (
    load_combined_dataset, clean_combined_responses,
    verify_tag_presence, update_id_attribute,
)
from pseudolabel_utils import deanonymize_dataframe_v1, deanonymize_dataframe_v2

# --- Config ---
DATA_SOURCE          = ["hermes"]
PSEUDO_LABEL         = False
COMBINE_SAMPLES      = True
ADD_FEW_SHOT_EXAMPLES = False

random.seed(2025)

# Data Loading and Merging
Load raw or pseudolabeled CSVs from each requested source, normalize column names, strip XML wrapper tags, and deduplicate. Sources not listed in `DATA_SOURCE` are skipped.

In [ ]:
final_df = load_combined_dataset(DATA_SOURCE, PSEUDO_LABEL)

In [ ]:
final_df = clean_combined_responses(final_df)

# Data Validation
Filter out rows whose generated report does not contain all PHI tags listed in `features_to_include`. Also remaps `hospital` → `place` to align with the XML tag vocabulary before checking.

## Tag Presence Check
Verify that every tag listed in `features_to_include` appears as a matched open/close pair in `default_response_1`. Rows with any missing tag are dropped; the charts below show the distribution before dropping.

In [ ]:
final_df['features_to_include'] = final_df['features_to_include'].str.replace("hospital", "place")
final_df['non_present_tags'] = final_df.apply(
    lambda row: verify_tag_presence(row["default_response_1"], row["features_to_include"].split(", ")),
    axis=1,
)

tag_counts = final_df['non_present_tags'].apply(len).value_counts().sort_index()
tag_names  = final_df['non_present_tags'].explode().value_counts()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

tag_counts.plot(kind='bar', ax=ax1)
ax1.set_title('Distribution of Number of Non-Present Tags')
ax1.set_xlabel('Number of Non-Present Tags')
ax1.set_ylabel('Count')
for i, v in enumerate(tag_counts):
    ax1.text(i, v, str(v), ha='center', va='bottom')

tag_names.plot(kind='bar', ax=ax2)
ax2.set_title('Distribution of Specific Non-Present Tags')
ax2.set_xlabel('Tag Name')
ax2.set_ylabel('Count')
for i, v in enumerate(tag_names):
    ax2.text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

final_df = final_df[final_df['non_present_tags'].apply(lambda x: len(x) == 0)]
final_df = final_df.reset_index(drop=True)
print(f"Number of rows after dropping: {len(final_df)}")

## Text Preprocessing
Normalize whitespace and encoding artifacts in both response columns using the shared `preprocess_text` pipeline step.

In [ ]:
final_df = preprocess_text(final_df, input_col="default_response_0", output_col="default_response_1")

## Composite Reports
Merge 10 randomly sampled individual reports into a single long-document sample. This generates ~5% of the current dataset size as composite examples and removes the constituent rows from the pool. The last 404 rows are reserved as the validation set and are never used as source material.

In [ ]:
if COMBINE_SAMPLES:
    composite_reports = []
    number_of_composite_reports = len(final_df) // 20  # 5% of the dataset size
    for i in tqdm(range(number_of_composite_reports)):
        random_indices = random.sample(range(len(final_df)), 10)
        random_indices = [idx for idx in random_indices if idx < len(final_df) - 404]  # reserved for testing
        composite_response_0 = []
        composite_response_1 = []
        for j, idx in enumerate(random_indices):
            composite_response_0.append(update_id_attribute(final_df.iloc[idx]["default_response_0"], f"{j}"))
            composite_response_1.append(update_id_attribute(final_df.iloc[idx]["default_response_1"], f"{j}"))
        composite_response_0 = "\n\n".join(composite_response_0)
        composite_response_1 = "\n\n".join(composite_response_1)
        composite_reports.append({"default_response_0": composite_response_0, "default_response_1": composite_response_1})

        for idx in random_indices:
            final_df = final_df.drop(idx)
        final_df.reset_index(inplace=True, drop=True)

    composite_df = pd.DataFrame(composite_reports)

    last_404_rows = final_df.iloc[-404:]
    rest_of_df    = final_df.iloc[:-404]

    final_df = pd.concat([rest_of_df, composite_df])
    final_df.sample(frac=1).reset_index(inplace=True, drop=True)
    final_df = pd.concat([final_df, last_404_rows])
    final_df.reset_index(inplace=True, drop=True)

    print(f"Number of composite reports: {len(composite_df)}")
    print(f"Average words in composite reports: {composite_df['default_response_0'].apply(lambda x: len(x.split())).mean():.0f}")
    display(composite_df.head())

## Data Augmentation
Apply stochastic PHI-level text transformations to `default_response_1` to increase training diversity. Transformations include date format randomization, name variations (misspellings, last-first swap), phone number formatting, and structural perturbations (paragraph joins, uppercase sentences, preposition drops).

In [ ]:
augmentor = TextAugmentor()
augmentor.add_augmentation(global_sanitize_date, 1)
augmentor.add_augmentation(global_date_random_shift, 1)
augmentor.add_augmentation(global_uniform_date, 0.2)
augmentor.add_augmentation(global_add_time_to_date, 0.1)
augmentor.add_augmentation(global_drop_prepositions, 1)
augmentor.add_augmentation(global_convert_yo, 1)
augmentor.add_augmentation(global_replace_patient_names, 0.5)
augmentor.add_augmentation(global_remove_on_before_date, 0.5)
augmentor.add_augmentation(global_join_paragraphs, 0.2)
augmentor.add_augmentation(global_drop_dot_in_title, 0.2)
augmentor.add_augmentation(global_drop_title_before_name, 0.2)
augmentor.add_augmentation(global_update_ids, 1)
augmentor.add_augmentation(global_remove_phis, 0.02)
augmentor.add_augmentation(global_remove_comma_before_suffixes, 0.5)
augmentor.add_augmentation([           # One-of date format augmentations
    (ymd_to_mdy, 0.15),
    (ymd_to_dmy, 0.15),
    (ymd_to_mdy2, 0.15),
    (ymd_remove_year, 0.15),
    (ymd_remove_year2, 0.15),
    (ymd_to_wordy, 0.15),
    (ymd_to_wordy2, 0.15),
    (ymd_to_wordy3, 0.15),
    (ymd_remove_day, 0.15),
    (ymd_remove_day_and_month, 0.30),
    (ymd_remove_day_and_year, 0.15),
    (ymd_to_wordy_short_month_no_year, 0.15),
    (ymd_to_wordy_comma_style, 0.15),
    (ymd_to_wordy_no_comma, 0.15),
    (ymd_to_wordy_short_month, 0.15),
], 0.95, "date")
augmentor.add_augmentation(fix_lastname_firstname, 0.2, "name")
augmentor.add_augmentation(introduce_misspelling_in_name, 0.1, "name")
augmentor.add_augmentation([           # One-of phone format augmentations
    (add_parantheses_to_phone_number, 0.1),
    (add_dash_to_phone_number, 0.1),
    (add_space_between_phone_number_parts, 0.1),
    (add_international_code_to_phone_number, 0.1),
], 0.95, "phone")
augmentor.add_augmentation(global_remove_spaces_inside_tags, 0.1)
augmentor.add_augmentation(global_make_sentence_uppercase, 0.1)

In [ ]:
final_df["default_response_1_augmented"] = final_df["default_response_1"].apply(augmentor.augment)

## Ground Truth v1 — Tag Stripping
Produce `final_v1` by stripping PHI XML tags from `default_response_1_augmented`. The annotated text (`anonymized`) is the model target; the stripped text (`original`) is the plain-text input.

In [ ]:
final_v1 = final_df.copy()
final_v1 = deanonymize_dataframe_v1(final_v1, "default_response_1_augmented", "default_response_1_deanonymized")
final_v1 = final_v1[["default_response_1_augmented", "default_response_1_deanonymized"]]
final_v1.columns = ["anonymized", "original"]
display(final_v1.head())

## Ground Truth v2 — Structured XML
Produce `final_v2` by converting inline PHI tags to the structured `<phi_entities>` XML format. `anonymized` holds the structured annotation; `original` is the plain-text input stripped of all tags.

In [ ]:
final_v2 = final_df.copy()
final_v2 = deanonymize_dataframe_v2(final_v2, "default_response_1_augmented", "xml_ground_truth", "native_text")
final_v2 = final_v2[["xml_ground_truth", "native_text"]]
final_v2.columns = ["anonymized", "original"]
display(final_v2.head())

## Few-Shot Examples
Optionally inject 1–3 in-context examples into a random 10% of training rows. Example rows are removed from the training pool after use. Skipped when `ADD_FEW_SHOT_EXAMPLES = False`.

In [ ]:
final_v2['examples'] = "NONE"
if ADD_FEW_SHOT_EXAMPLES:
    num_examples = len(final_v2) // 10  # 10% of the dataset size
    indices_with_examples = random.choices(final_v2.index[:-404], k=num_examples)
    examples_indices = []
    for index in indices_with_examples:
        examples = []
        for _ in range(random.choice([1, 2, 3])):
            example_idx = None
            while example_idx is None:
                example_idx = random.choice(final_v2.index[:-404])
                if example_idx in indices_with_examples:
                    example_idx = None
                    continue
                original = final_v2.at[example_idx, "original"]
                anonymized_example = final_v2.at[example_idx, "anonymized"]
                if len(original.split()) > 500:
                    example_idx = None
                    continue
                anonymized_example = anonymized_example.replace("<phi_entities>\n", "").replace("\n</phi_entities>", "")
                examples.append({"original": original, "anonymized": anonymized_example})
                examples_indices.append(example_idx)
        final_v2.at[index, "examples"] = json.dumps(examples)

    final_v2.drop(index=examples_indices, inplace=True)
    final_v2.reset_index(inplace=True, drop=True)

## Curriculum Ordering and Final Shuffle
Sort the training split by annotation line count (short → long) for curriculum learning. Visualize the resulting distribution, then shuffle the training split before recombining with the fixed validation set.

In [ ]:
train_df = final_v2[:-404].copy()
val_df   = final_v2[-404:].copy()

train_df['line_count'] = train_df['anonymized'].str.count('\n')
train_df = train_df.sort_values('line_count').drop('line_count', axis=1)

final_v2_lines = pd.concat([train_df, val_df]).reset_index(drop=True)
line_counts = final_v2_lines['anonymized'].apply(lambda x: len(x.split("\n")))

plt.figure(figsize=(10, 6))
plt.axvspan(len(line_counts) - 404, len(line_counts), color='lightgreen', alpha=0.3)
plt.axvspan(0, len(line_counts[line_counts == 12]), color='lightcoral', alpha=0.3)
plt.plot(range(len(line_counts)), line_counts, color='skyblue')
plt.title('Line Count Distribution Across Samples')
plt.xlabel('Sample Index')
plt.ylabel('Number of Lines')
plt.grid(True, alpha=0.75)
plt.show()

In [ ]:
# Shuffle training split and recombine with validation
train_df = train_df.sample(frac=1).reset_index(drop=True)
final_v2 = pd.concat([train_df, val_df]).reset_index(drop=True)

print("Empty outputs sample:")
display(final_v2[final_v2["anonymized"] == "<phi_entities>\n</phi_entities>"].head())
print(f"Final number of rows: {len(final_v2)}")

# Save Datasets
Serialize the training and test splits for both versions of the ground truth (v1 and v2) into CSV files.

In [ ]:
final_v1[:-404].to_csv("../data/v1_train.csv", index=False)
final_v1[-404:].to_csv("../data/v1_test.csv", index=False)
final_v2[:-404].to_csv("../data/train.csv", index=False)
final_v2[-404:].to_csv("../data/test.csv", index=False)